### BERT Soft-Label Learning-Rate and Epoch Tuning

This notebook extends `BERT_Soft_Label_Experiment_Runner.ipynb` for a more intensive fine-tuning pass. It keeps the same dataset construction, soft-label objective, threshold tuning, and evaluation metrics, then adds a learning-rate sweep, longer training, validation-loss tracking, and plots for selecting the best run.


In [ ]:
from __future__ import annotations

import json
import math
import random
from dataclasses import asdict, dataclass, replace
from pathlib import Path
from typing import Iterable

# display makes tables render nicely in Jupyter and keeps dry runs explicit.
try:
    from IPython.display import display
except ModuleNotFoundError:
    def display(obj):
        print(obj)

# Core scientific Python stack.
# numpy/pandas handle the annotation matrices and saved result tables.
import numpy as np
import pandas as pd

# PyTorch is used for the model, dataloaders, custom loss, and training loop.
import torch
import torch.nn.functional as F
from torch import nn
from torch.optim import AdamW
from torch.utils.data import DataLoader, Dataset

# Hugging Face transformers supplies the tokenizer, pretrained encoders, and learning-rate scheduler.
from transformers import (
    AutoModel,
    AutoTokenizer,
    get_constant_schedule_with_warmup,
    get_cosine_schedule_with_warmup,
    get_linear_schedule_with_warmup,
)

# Metrics for soft-label evaluation and thresholded multi-label evaluation.
from scipy.stats import pearsonr, spearmanr
from sklearn.metrics import (
    average_precision_score,
    classification_report,
    f1_score,
    hamming_loss,
    mean_absolute_error,
    mean_squared_error,
)
from sklearn.model_selection import train_test_split

# If not installed the notebook uses random splits.
try:
    from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit
    HAS_ITERSTRAT = True
except ModuleNotFoundError:
    MultilabelStratifiedShuffleSplit = None
    HAS_ITERSTRAT = False

print('iterative-stratification available:', HAS_ITERSTRAT)


Project folders, output folder, and the eight reappraisal strategy labels.


In [ ]:
PROJECT_DIR = next(
    (root for root in (Path.cwd(), *Path.cwd().parents)
     if (root / 'data' / 'annotate-lime.csv').exists()),
    None,
)
if PROJECT_DIR is None:
    raise FileNotFoundError('Could not locate the project root containing data/annotate-lime.csv.')
DATA_DIR = PROJECT_DIR / 'data'

# Keep tuning artifacts separate from the first experiment so checkpoints and CSVs are not overwritten.
OUTPUT_DIR = PROJECT_DIR / 'Code' / 'BERT_models' / 'experiments' / 'lr_epoch_tuning'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# 8 strategies
REAPPRAISAL_TECHNIQUES = ['[GM]', '[PR]', '[A]', '[EJ]', '[N]', '[UC]', '[V]', '[H]']
LABEL_COLS = [tech.strip('[]') for tech in REAPPRAISAL_TECHNIQUES]

LABEL_NAMES = {
    'GM': 'growth mindset',
    'PR': 'positive repurposing',
    'A': 'aberration',
    'EJ': 'external justification',
    'N': 'normalizing',
    'UC': 'unwarranted conclusions',
    'V': 'validation',
    'H': 'humor',
}

print('Project directory:', PROJECT_DIR)
print('Output directory:', OUTPUT_DIR)
print('Labels:', LABEL_COLS)


### Model Configurations

The default sweep uses longer training than the original notebook. Early stopping is still active, so the model can train longer when validation loss keeps improving without forcing unnecessary epochs after convergence.


In [ ]:
@dataclass(frozen=True)
class ModelConfig:
    experiment_name: str
    model_name: str
    batch_size: int = 8
    learning_rate: float = 1e-5
    weight_decay: float = 0.01
    dropout: float = 0.4
    max_length: int = 256
    num_epochs: int = 30
    patience: int = 6
    scheduler_type: str = 'linear'
    local_files_only: bool = False
    pooling: str = 'auto'


# Base encoder settings. The learning-rate sweep below creates run-specific copies.
BERT_BASE_CONFIG = ModelConfig(
    experiment_name='bert_base_uncased',
    model_name='bert-base-uncased',
    batch_size=8,
    dropout=0.4,
    local_files_only=True,
)

BERT_LARGE_CONFIG = ModelConfig(
    experiment_name='bert_large_uncased',
    model_name='google-bert/bert-large-uncased',
    batch_size=4,
    dropout=0.5,
    local_files_only=False,
)

MODERNBERT_CONFIG = ModelConfig(
    experiment_name='modernbert_base',
    model_name='answerdotai/ModernBERT-base',
    batch_size=8,
    dropout=0.4,
    local_files_only=False,
)

ROBERTA_BASE_CONFIG = ModelConfig(
    experiment_name='roberta_base_cls',
    model_name='FacebookAI/roberta-base',
    batch_size=8,
    dropout=0.4,
    local_files_only=False,
    pooling='first_token',
)

ROBERTA_LARGE_CONFIG = ModelConfig(
    experiment_name='roberta_large_cls',
    model_name='FacebookAI/roberta-large',
    batch_size=4,
    dropout=0.5,
    local_files_only=False,
    pooling='first_token',
)


def make_roberta_large_tapt_config(checkpoint_dir: Path) -> ModelConfig:
    return replace(
        ROBERTA_LARGE_CONFIG,
        experiment_name='roberta_large_tapt_cls',
        model_name=str(checkpoint_dir.resolve()),
        local_files_only=True,
    )

# Research workflow: sweep conservative rates around the previously successful values.
# For the best final result, keep all three encoders. For a quicker pilot, set this to [BERT_BASE_CONFIG].
BASE_MODEL_CONFIGS = [
    BERT_BASE_CONFIG,
    BERT_LARGE_CONFIG,
    MODERNBERT_CONFIG,
    ROBERTA_BASE_CONFIG,
    ROBERTA_LARGE_CONFIG,
]

LEARNING_RATES_BY_MODEL = {
    'bert_base_uncased': [5e-6, 1e-5, 2e-5, 3e-5],
    'bert_large_uncased': [2e-6, 5e-6, 1e-5],
    'modernbert_base': [5e-6, 1e-5, 2e-5],
    'roberta_base_cls': [5e-6, 1e-5, 2e-5],
    'roberta_large_cls': [5e-6, 1e-5],
}

EPOCHS_TO_TRY = 30
EARLY_STOPPING_PATIENCE = 6

BASE_MODEL_CONFIGS


In [ ]:
# random seeds
def set_seed(seed: int) -> None:

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


#Use CUDA
def get_device() -> torch.device:
    if torch.cuda.is_available():
        return torch.device('cuda')
    if torch.backends.mps.is_available():
        return torch.device('mps')
    return torch.device('cpu')


device = get_device()
print('Using device:', device)


### Load The Reappraisal Dataset

Building the  modeling table. For every strategy it looks at the corresponding annotation & computes:

- positive votes
- number of valid votes
- soft score = positive votes / valid votes

The model sees `SITUATION` and `REAPPRAISAL` as text input. The target is the eight soft-label scores.

In [ ]:
def load_reappraisal_dataset(data_dir: Path = DATA_DIR) -> pd.DataFrame:
    texts = pd.read_csv(data_dir / 'annotate-lime.csv')

    soft_labels = pd.DataFrame()
    vote_counts = pd.DataFrame()
    positive_votes = pd.DataFrame() # if strategy was given 1 for present

    for tech in REAPPRAISAL_TECHNIQUES:
        label = tech.strip('[]')

        # Each strategy has one semicolon-separated annotation matrix.
        ann = pd.read_csv(data_dir / f'annotations-only-{tech}.csv', sep=';')
        ann = ann.apply(pd.to_numeric, errors='coerce')

        # raw vote counts and the observed annotator proportion.
        positive_votes[f'{label}_positive_votes'] = ann.sum(axis=0).values
        vote_counts[f'{label}_n_votes'] = ann.count(axis=0).values
        soft_labels[label] = ann.mean(axis=0).values

    dataset = pd.concat(
        [
            texts[['key', 'SITUATION', 'REAPPRAISAL']].reset_index(drop=True),
            soft_labels.reset_index(drop=True),
            positive_votes.reset_index(drop=True),
            vote_counts.reset_index(drop=True),
        ],
        axis=1,
    )

    dataset['vignette_id'] = pd.factorize(dataset['SITUATION'])[0]

    # Sanity check: the soft labels should exactly equal positive_votes / n_votes.
    expected_scores = (
        dataset[[f'{label}_positive_votes' for label in LABEL_COLS]].to_numpy()
        / dataset[[f'{label}_n_votes' for label in LABEL_COLS]].to_numpy()
    )
    if not np.allclose(dataset[LABEL_COLS].to_numpy(), expected_scores, equal_nan=True):
        raise ValueError('Soft labels do not match positive_votes / n_votes.')

    return dataset


# Secondary binary reports use three interpretations of what it means for a
# strategy to be "present". Training still uses the soft labels above.
BINARY_TARGET_SCHEMES = {
    'weak_presence': 'at least one annotator marked the strategy present',
    'majority': 'more than half of valid annotators marked the strategy present',
    'strong_presence': 'at least 80% of valid annotators marked the strategy present',
}


def binary_target_matrix(df: pd.DataFrame, scheme: str = 'majority') -> np.ndarray:

    positive_votes = df[[f'{label}_positive_votes' for label in LABEL_COLS]].to_numpy()
    n_votes = df[[f'{label}_n_votes' for label in LABEL_COLS]].to_numpy()
    soft_scores = positive_votes / np.clip(n_votes, 1, None)

    if scheme == 'weak_presence':
        return (positive_votes >= 1).astype(int)
    if scheme == 'majority':
        return (positive_votes > (n_votes / 2)).astype(int)
    if scheme == 'strong_presence':
        return (soft_scores >= 0.80).astype(int)
    raise ValueError(f'Unknown binary target scheme: {scheme}')


def binary_targets(df: pd.DataFrame, scheme: str = 'majority') -> pd.DataFrame:
    #Return binary target labels as a DF with label columns
    return pd.DataFrame(binary_target_matrix(df, scheme), columns=LABEL_COLS, index=df.index)


dataset = load_reappraisal_dataset()
print('Dataset shape:', dataset.shape)
print('Number of situations:', dataset['SITUATION'].nunique())
display(dataset[['key', 'SITUATION', 'REAPPRAISAL'] + LABEL_COLS].head())

# Show how many positives each binary definition creates.
binary_count_summary = pd.concat(
    [binary_targets(dataset, scheme).sum().rename(scheme) for scheme in BINARY_TARGET_SCHEMES],
    axis=1,
)
display(binary_count_summary)


## Train/Validation/Test Splits

Iterative multilabel stratification to preserve rare labels across train, validation, and test sets. Else notebook uses seeded random split.

In [ ]:
def multilabel_split_indices(
    df: pd.DataFrame,
    y_binary: pd.DataFrame,
    test_size: float,
    seed: int,
) -> tuple[np.ndarray, np.ndarray]:
 #Return train/test indices using multilabel stratification

    if HAS_ITERSTRAT:
        splitter = MultilabelStratifiedShuffleSplit(
            n_splits=1,
            test_size=test_size,
            random_state=seed,
        )
        return next(splitter.split(df, y_binary))

    print(
        'Warning: iterative stratification not installed; '
        'using a seeded random split.'
    )
    indices = np.arange(len(df))
    return train_test_split(
        indices,
        test_size=test_size,
        random_state=seed,
        shuffle=True,
    )


def make_random_split(
    dataset: pd.DataFrame,
    seed: int,
    split_scheme: str = 'majority',
    test_size: float = 0.15,
    val_size: float = 0.15,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    #70/15/15 train/validation/test split. first split creates train vs. temporary data. The second split divides the temporary data equally into validation and test.


    temp_size = test_size + val_size
    train_idx, temp_idx = multilabel_split_indices(
        dataset,
        binary_targets(dataset, split_scheme),
        test_size=temp_size,
        seed=seed,
    )

    train_df = dataset.iloc[train_idx].reset_index(drop=True)
    temp_df = dataset.iloc[temp_idx].reset_index(drop=True)

    test_fraction_of_temp = test_size / temp_size
    val_idx, test_idx = multilabel_split_indices(
        temp_df,
        binary_targets(temp_df, split_scheme),
        test_size=test_fraction_of_temp,
        seed=seed,
    )

    val_df = temp_df.iloc[val_idx].reset_index(drop=True)
    test_df = temp_df.iloc[test_idx].reset_index(drop=True)
    return train_df, val_df, test_df


def make_leave_one_vignette_splits(
    dataset: pd.DataFrame,
    seed: int,
    split_scheme: str = 'majority',
    val_size: float = 0.15,
) -> list[tuple[str, pd.DataFrame, pd.DataFrame, pd.DataFrame]]:
    #leave one out

    splits = []
    for situation, test_df in dataset.groupby('SITUATION', sort=False):
        train_val_df = dataset.loc[dataset['SITUATION'] != situation].reset_index(drop=True)
        test_df = test_df.reset_index(drop=True)

        train_idx, val_idx = multilabel_split_indices(
            train_val_df,
            binary_targets(train_val_df, split_scheme),
            test_size=val_size,
            seed=seed,
        )
        train_df = train_val_df.iloc[train_idx].reset_index(drop=True)
        val_df = train_val_df.iloc[val_idx].reset_index(drop=True)
        splits.append((situation, train_df, val_df, test_df))

    return splits


train_df, val_df, test_df = make_random_split(dataset, seed=42, split_scheme='majority')
print('Example random split sizes:', len(train_df), len(val_df), len(test_df))


### PyTorch Dataset

The dataset object turns each row into tensors.

- tokenizer outputs for SITUATION and REAPPRAISAL.
- soft labels for the eight strategy scores
- positive vote counts
- valid vote counts


In [ ]:
class ReappraisalDataset(Dataset):
    def __init__(self, df: pd.DataFrame, tokenizer, max_length: int):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self) -> int:
        return len(self.df)

    def __getitem__(self, idx: int) -> dict[str, torch.Tensor]:
        row = self.df.iloc[idx]

        # Passing situation and reappraisal as a text pair lets BERT use segment information
        # for models that support token_type_ids.
        encoding = self.tokenizer(
            str(row['SITUATION']),
            str(row['REAPPRAISAL']),
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            return_tensors='pt',
        )

        # Remove the extra batch dimension added by return_tensors='pt'.
        item = {key: value.squeeze(0) for key, value in encoding.items()}

        # Soft target: observed annotator proportion for each strategy.
        item['labels'] = torch.tensor(row[LABEL_COLS].values.astype('float32'))

        # Raw annotation counts used by binary cross-entropy + sigmoid logits
        item['positive_votes'] = torch.tensor(
            row[[f'{label}_positive_votes' for label in LABEL_COLS]].values.astype('float32')
        )
        item['n_votes'] = torch.tensor(
            row[[f'{label}_n_votes' for label in LABEL_COLS]].values.astype('float32')
        )
        return item


### Model Definition

Model wraps Hugging Face encoder with a shared eight-output prediction head.

The architecture is:
Situation + Reappraisal -> pretrained encoder -> pooled representation -> dropout -> linear layer -> 8 logits

The logits become probabilities after applying sigmoid.

In [ ]:
class EncoderForReappraisalSoftLabels(nn.Module):
    def __init__(self, config: ModelConfig, num_labels: int = len(LABEL_COLS)):
        super().__init__()

        # AutoModel lets this same class load BERT, BERT-large, ModernBERT, etc.
        self.encoder = AutoModel.from_pretrained(
            config.model_name,
            local_files_only=config.local_files_only,
        )
        if config.pooling not in {'auto', 'first_token'}:
            raise ValueError("pooling must be 'auto' or 'first_token'.")
        self.pooling = config.pooling
        self.dropout = nn.Dropout(config.dropout)
        self.regressor = nn.Linear(self.encoder.config.hidden_size, num_labels)

    def forward(self, **encoder_inputs) -> torch.Tensor:
        outputs = self.encoder(**encoder_inputs)

        if self.pooling == 'first_token':
            pooled = outputs.last_hidden_state[:, 0]
        else:
            pooled = getattr(outputs, 'pooler_output', None)
            if pooled is None:
                pooled = outputs.last_hidden_state[:, 0]

        return self.regressor(self.dropout(pooled))


### Loss Functions

The recommended loss binary cross-entropy with sigmoid logits.


k = number of annotators who marked the strategy present
& n = number of valid annotator judgments


Better than training only on the mean score because it preserves that labels came from multiple judgments.

Also possible bce_soft and mse as comparison option.

In [ ]:
def split_batch(batch: dict[str, torch.Tensor], device: torch.device):
    labels = batch['labels'].to(device)
    positive_votes = batch['positive_votes'].to(device)
    n_votes = batch['n_votes'].to(device)

    # Keep every tokenizer field returned by the tokenizer.
    # This avoids mismatches between training and prediction for models with token_type_ids.
    encoder_inputs = {
        key: value.to(device)
        for key, value in batch.items()
        if key not in {'labels', 'positive_votes', 'n_votes'}
    }
    return encoder_inputs, labels, positive_votes, n_votes


def binomial_nll_loss(logits: torch.Tensor, positive_votes: torch.Tensor, n_votes: torch.Tensor) -> torch.Tensor:
    #binary cross-entropy with sigmoid logits

    targets = positive_votes / n_votes.clamp_min(1.0)
    per_vote_bce = F.binary_cross_entropy_with_logits(logits, targets, reduction='none')
    return (per_vote_bce * n_votes).sum() / n_votes.sum().clamp_min(1.0)


def compute_loss(
    logits: torch.Tensor,
    labels: torch.Tensor,
    positive_votes: torch.Tensor,
    n_votes: torch.Tensor,
    loss_mode: str,
) -> torch.Tensor:
    #Dispatch to loss function

    if loss_mode == 'binomial_nll':
        return binomial_nll_loss(logits, positive_votes, n_votes)
    if loss_mode == 'bce_soft':
        return F.binary_cross_entropy_with_logits(logits, labels)
    if loss_mode == 'mse':
        return F.mse_loss(torch.sigmoid(logits), labels)
    raise ValueError("loss_mode must be 'binomial_nll', 'bce_soft', or 'mse'.")


### Training And Prediction Helpers

Training epoch, validation loss, and prediction.

Optimizer uses weight decay when appropriate: biases and layer-norm weights are excluded from weight decay.

In [ ]:
def build_optimizer(model: nn.Module, learning_rate: float, weight_decay: float) -> AdamW:
    #AdamW with no weight decay on bias and layer-normalization parameters

    no_decay = ('bias', 'LayerNorm.weight', 'layer_norm.weight')
    grouped_params = [
        {
            'params': [
                p for n, p in model.named_parameters()
                if p.requires_grad and not any(nd in n for nd in no_decay)
            ],
            'weight_decay': weight_decay,
        },
        {
            'params': [
                p for n, p in model.named_parameters()
                if p.requires_grad and any(nd in n for nd in no_decay)
            ],
            'weight_decay': 0.0,
        },
    ]
    return AdamW(grouped_params, lr=learning_rate)


def build_scheduler(optimizer, total_training_steps: int, scheduler_type: str):
    # All scheduler comparisons use the same 10% warmup so the decay shape is the only change.
    warmup_steps = int(0.10 * total_training_steps)
    if scheduler_type == 'linear':
        return get_linear_schedule_with_warmup(
            optimizer,
            num_warmup_steps=warmup_steps,
            num_training_steps=total_training_steps,
        )
    if scheduler_type == 'cosine':
        return get_cosine_schedule_with_warmup(
            optimizer,
            num_warmup_steps=warmup_steps,
            num_training_steps=total_training_steps,
        )
    if scheduler_type == 'constant_with_warmup':
        return get_constant_schedule_with_warmup(
            optimizer,
            num_warmup_steps=warmup_steps,
        )
    raise ValueError(f'Unknown scheduler_type: {scheduler_type}')


def train_one_epoch(model, dataloader, optimizer, scheduler, device, loss_mode: str) -> float:
    #Train the model for one pass over the training data

    model.train()
    total_loss = 0.0

    for batch in dataloader:
        optimizer.zero_grad(set_to_none=True)
        encoder_inputs, labels, positive_votes, n_votes = split_batch(batch, device)
        logits = model(**encoder_inputs)
        loss = compute_loss(logits, labels, positive_votes, n_votes, loss_mode)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()

    return total_loss / len(dataloader)


def evaluate_loss(model, dataloader, device, loss_mode: str) -> float:
    """Compute validation/test loss without updating model weights."""

    model.eval()
    total_loss = 0.0

    with torch.no_grad():
        for batch in dataloader:
            encoder_inputs, labels, positive_votes, n_votes = split_batch(batch, device)
            logits = model(**encoder_inputs)
            loss = compute_loss(logits, labels, positive_votes, n_votes, loss_mode)
            total_loss += loss.item()

    return total_loss / len(dataloader)


def get_predictions(model, dataloader, device) -> tuple[np.ndarray, np.ndarray]:
    #Return sigmoid probabilities and true soft labels for a dataloader

    model.eval()
    all_probs = []
    all_labels = []

    with torch.no_grad():
        for batch in dataloader:
            encoder_inputs, labels, _, _ = split_batch(batch, device)
            logits = model(**encoder_inputs)
            all_probs.append(torch.sigmoid(logits).cpu().numpy())
            all_labels.append(labels.cpu().numpy())

    return np.vstack(all_probs), np.vstack(all_labels)


### Evaluation Metrics
- MAE
- RMSE
- Pearson correlation
- Spearman correlation

Thresholded F1 only for understandability of binary classification.

In [ ]:
def safe_corr(fn, y_true: np.ndarray, y_score: np.ndarray) -> float:
    if np.std(y_true) == 0 or np.std(y_score) == 0:
        return np.nan
    return float(fn(y_true, y_score).statistic)


def soft_label_metrics(y_true: np.ndarray, y_score: np.ndarray) -> pd.DataFrame:
    rows = []
    for i, label in enumerate(LABEL_COLS):
        yt = y_true[:, i]
        ys = y_score[:, i]
        rows.append({
            'label': label,
            'strategy': LABEL_NAMES[label],
            'mae': mean_absolute_error(yt, ys),
            'rmse': mean_squared_error(yt, ys) ** 0.5,
            'pearson': safe_corr(pearsonr, yt, ys),
            'spearman': safe_corr(spearmanr, yt, ys),
            'mean_true': yt.mean(),
            'mean_pred': ys.mean(),
        })
    return pd.DataFrame(rows)


def macro_soft_summary(metrics_df: pd.DataFrame) -> dict[str, float]:
    return {
        'macro_mae': float(metrics_df['mae'].mean()),
        'macro_rmse': float(metrics_df['rmse'].mean()),
        'mean_pearson': float(metrics_df['pearson'].mean(skipna=True)),
        'mean_spearman': float(metrics_df['spearman'].mean(skipna=True)),
    }


### Threshold Tuning And Binary Metrics

The model is trained on soft labels. After training also three secondary binary labels:

- weak presence: at least one annotator marked the strategy present
- majority: more than half of valid annotators marked the strategy present
- strong presence: at least 80% of valid annotators marked the strategy present

For each, prediction thresholds are tuned on the validation set and evaluated on the test set.


In [ ]:
def tune_thresholds(
    y_true_binary: np.ndarray,
    y_score: np.ndarray,
    grid: Iterable[float] | None = None,
) -> tuple[dict[str, float], pd.DataFrame]:
    #prediction threshold per label using validation F1


    if grid is None:
        grid = np.arange(0.05, 0.96, 0.05)

    thresholds = {}
    tuning_rows = []

    for i, label in enumerate(LABEL_COLS):
        # If validation contains only one class for a rare label, F1 tuning is undefined.
        # In that case, keep the neutral 0.5 threshold and record the issue.
        if len(np.unique(y_true_binary[:, i])) < 2:
            thresholds[label] = 0.5
            tuning_rows.append({
                'label': label,
                'threshold': 0.5,
                'validation_f1': np.nan,
                'note': 'validation set has one class',
            })
            continue

        best_threshold = 0.5
        best_f1 = -1.0
        for threshold in grid:
            pred = (y_score[:, i] >= threshold).astype(int)
            score = f1_score(y_true_binary[:, i], pred, zero_division=0)
            if score > best_f1:
                best_f1 = score
                best_threshold = float(threshold)

        thresholds[label] = best_threshold
        tuning_rows.append({
            'label': label,
            'threshold': best_threshold,
            'validation_f1': best_f1,
            'note': '',
        })

    return thresholds, pd.DataFrame(tuning_rows)


def apply_thresholds(scores: np.ndarray, thresholds: dict[str, float]) -> np.ndarray:
    """Apply per-label prediction thresholds to the model score matrix."""

    threshold_array = np.array([thresholds[label] for label in LABEL_COLS])
    return (scores >= threshold_array).astype(int)


def evaluate_multilabel(
    y_true_binary: np.ndarray,
    scores: np.ndarray,
    thresholds: dict[str, float],
) -> tuple[dict[str, float], pd.DataFrame, pd.DataFrame, str, np.ndarray]:
    #Evaluate thresholded predictions as a conventional multi-label task

    y_pred = apply_thresholds(scores, thresholds)

    metrics = {
        'micro_f1': f1_score(y_true_binary, y_pred, average='micro', zero_division=0),
        'macro_f1': f1_score(y_true_binary, y_pred, average='macro', zero_division=0),
        'samples_f1': f1_score(y_true_binary, y_pred, average='samples', zero_division=0),
        'hamming_loss': hamming_loss(y_true_binary, y_pred),
    }

    ap_rows = []
    count_rows = []
    for i, label in enumerate(LABEL_COLS):
        ap = np.nan
        if len(np.unique(y_true_binary[:, i])) >= 2:
            ap = average_precision_score(y_true_binary[:, i], scores[:, i])
        ap_rows.append({'label': label, 'average_precision': ap})
        count_rows.append({
            'label': label,
            'true_positive_items': int(y_true_binary[:, i].sum()),
            'predicted_positive_items': int(y_pred[:, i].sum()),
        })

    report = classification_report(y_true_binary, y_pred, target_names=LABEL_COLS, zero_division=0)
    return metrics, pd.DataFrame(ap_rows), pd.DataFrame(count_rows), report, y_pred


### Saving Results

 checkpoint, metrics, predictions, and summary files.


In [ ]:
def sanitize_for_json(value):
    #Convert NumPy values and NaNs into JSON values

    if isinstance(value, dict):
        return {key: sanitize_for_json(inner) for key, inner in value.items()}
    if isinstance(value, list):
        return [sanitize_for_json(inner) for inner in value]
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        if math.isnan(float(value)):
            return None
        return float(value)
    if isinstance(value, float) and math.isnan(value):
        return None
    return value


def write_json(path: Path, payload: dict) -> None:
    """Write metrics as strict JSON."""

    path.write_text(json.dumps(sanitize_for_json(payload), indent=2))


### Run One Experiment

 Train one model on one split, evaluate soft-label prediction on the test set, then report weak/majority/strong binary metrics as secondary analyses.


In [ ]:
def validate_tapt_checkpoint_for_run(
    config: ModelConfig,
    run_id: str,
    train_df: pd.DataFrame,
) -> None:
    checkpoint_dir = Path(config.model_name)
    if not checkpoint_dir.is_dir():
        return

    manifest_path = checkpoint_dir / 'tapt_manifest.json'
    if not manifest_path.exists():
        if 'tapt' in checkpoint_dir.name.lower():
            raise ValueError(f'TAPT checkpoint has no manifest: {checkpoint_dir}')
        return
    if checkpoint_dir.name.endswith('_smoke_test'):
        raise ValueError('Smoke-test TAPT checkpoints cannot be used downstream.')

    manifest = json.loads(manifest_path.read_text())
    if manifest.get('best_epoch') is None:
        raise ValueError(f'TAPT checkpoint has no valid best epoch: {checkpoint_dir}')

    assignments_path = checkpoint_dir / 'mlm_split_assignments.csv'
    if not assignments_path.exists():
        raise ValueError(f'TAPT checkpoint has no split assignments: {checkpoint_dir}')
    checkpoint_train_keys = set(
        pd.read_csv(assignments_path)['key'].astype(str)
    )
    downstream_train_keys = set(train_df['key'].astype(str))
    if checkpoint_train_keys != downstream_train_keys:
        overlap = len(checkpoint_train_keys & downstream_train_keys)
        raise ValueError(
            'TAPT checkpoint training keys do not match the downstream training split: '
            f'{overlap} of {len(downstream_train_keys)} keys overlap.'
        )

    checkpoint_run_id = manifest.get('downstream_run_id')
    if checkpoint_run_id is None:
        checkpoint_run_id = manifest.get('config', {}).get('downstream_run_id')

    if checkpoint_run_id is not None:
        if checkpoint_run_id != run_id:
            raise ValueError(
                f'TAPT checkpoint was built for {checkpoint_run_id}, not {run_id}.'
            )
    else:
        checkpoint_seed = manifest.get('config', {}).get('seed')
        if checkpoint_seed is None or not run_id.endswith(f'_seed{checkpoint_seed}'):
            raise ValueError(f'TAPT checkpoint seed does not match downstream run {run_id}.')
        if run_id.startswith('leave_vignette_'):
            raise ValueError(
                'Legacy TAPT manifest cannot verify the held-out vignette; regenerate it.'
            )

    model_files = [checkpoint_dir / 'model.safetensors', checkpoint_dir / 'pytorch_model.bin']
    if not any(path.exists() for path in model_files):
        raise ValueError(f'TAPT checkpoint contains no saved encoder weights: {checkpoint_dir}')


def run_single_experiment(
    config: ModelConfig,
    train_df: pd.DataFrame,
    val_df: pd.DataFrame,
    test_df: pd.DataFrame,
    run_id: str,
    output_dir: Path = OUTPUT_DIR,
    loss_mode: str = 'binomial_nll',
    device: torch.device | None = None,
) -> dict:
    #Train and evaluate one model configuration on one data split.

    output_dir.mkdir(parents=True, exist_ok=True)
    device = device or get_device()
    validate_tapt_checkpoint_for_run(config, run_id, train_df)

    # Tokenizer and dataset are created inside the run so each model uses its own tokenizer.
    tokenizer = AutoTokenizer.from_pretrained(
        config.model_name,
        local_files_only=config.local_files_only,
    )
    train_loader = DataLoader(
        ReappraisalDataset(train_df, tokenizer, config.max_length),
        batch_size=config.batch_size,
        shuffle=True,
    )
    val_loader = DataLoader(
        ReappraisalDataset(val_df, tokenizer, config.max_length),
        batch_size=config.batch_size,
        shuffle=False,
    )
    test_loader = DataLoader(
        ReappraisalDataset(test_df, tokenizer, config.max_length),
        batch_size=config.batch_size,
        shuffle=False,
    )

    model = EncoderForReappraisalSoftLabels(config).to(device)
    optimizer = build_optimizer(model, config.learning_rate, config.weight_decay)

    total_training_steps = len(train_loader) * config.num_epochs
    scheduler = build_scheduler(optimizer, total_training_steps, config.scheduler_type)

    # Checkpoint name includes both model and run id, so different experiments do not overwrite each other.
    checkpoint_path = output_dir / f'{config.experiment_name}_{run_id}_best_model.pt'
    best_val_loss = float('inf')
    best_epoch = None
    bad_epochs = 0
    history = []

    for epoch in range(1, config.num_epochs + 1):
        train_loss = train_one_epoch(model, train_loader, optimizer, scheduler, device, loss_mode)
        val_loss = evaluate_loss(model, val_loader, device, loss_mode)
        history.append({
            'experiment_name': config.experiment_name,
            'model_name': config.model_name,
            'run_id': run_id,
            'learning_rate': config.learning_rate,
            'scheduler_type': config.scheduler_type,
            'num_epochs_planned': config.num_epochs,
            'epoch': epoch,
            'train_loss': train_loss,
            'val_loss': val_loss,
        })
        print(
            f'{config.experiment_name} | {run_id} | {config.scheduler_type} | epoch {epoch:02d} '
            f'| train_loss={train_loss:.4f} | val_loss={val_loss:.4f}'
        )

        # Save the best model according to validation loss.
        if val_loss < best_val_loss - 1e-4:
            best_val_loss = val_loss
            best_epoch = epoch
            bad_epochs = 0
            torch.save(model.state_dict(), checkpoint_path)
        else:
            bad_epochs += 1
            if bad_epochs >= config.patience:
                print(f'Early stopping at epoch {epoch}; best epoch was {best_epoch}.')
                break

    # Reload the best checkpoint before computing validation/test predictions.
    model.load_state_dict(torch.load(checkpoint_path, map_location=device))
    val_probs, val_labels = get_predictions(model, val_loader, device)
    test_probs, test_labels = get_predictions(model, test_loader, device)

    # Soft-label metrics are the primary test metrics.
    soft_metrics = soft_label_metrics(test_labels, test_probs)

    # Binary metrics are secondary. We report three human-label definitions:
    # weak presence, exact majority, and strong presence.
    binary_scheme_metrics = {}
    threshold_tables = []
    count_tables = []
    average_precision_tables = []
    tuned_predictions_by_scheme = {}

    for scheme_name, scheme_description in BINARY_TARGET_SCHEMES.items():
        val_binary = binary_target_matrix(val_df, scheme_name)
        test_binary = binary_target_matrix(test_df, scheme_name)

        thresholds, threshold_df = tune_thresholds(val_binary, val_probs)
        threshold_df.insert(0, 'scheme', scheme_name)
        threshold_df.insert(1, 'scheme_description', scheme_description)
        threshold_tables.append(threshold_df)

        fixed_metrics, fixed_ap, fixed_counts, fixed_report, fixed_pred = evaluate_multilabel(
            test_binary,
            test_probs,
            {label: 0.5 for label in LABEL_COLS},
        )
        tuned_metrics, tuned_ap, tuned_counts, tuned_report, tuned_pred = evaluate_multilabel(
            test_binary,
            test_probs,
            thresholds,
        )

        fixed_ap.insert(0, 'scheme', scheme_name)
        tuned_ap.insert(0, 'scheme', scheme_name)
        fixed_ap.insert(1, 'threshold_type', 'fixed_0_5')
        tuned_ap.insert(1, 'threshold_type', 'validation_tuned')
        average_precision_tables.extend([fixed_ap, tuned_ap])

        fixed_counts.insert(0, 'scheme', scheme_name)
        tuned_counts.insert(0, 'scheme', scheme_name)
        fixed_counts.insert(1, 'threshold_type', 'fixed_0_5')
        tuned_counts.insert(1, 'threshold_type', 'validation_tuned')
        count_tables.extend([fixed_counts, tuned_counts])

        tuned_predictions_by_scheme[scheme_name] = tuned_pred
        binary_scheme_metrics[scheme_name] = {
            'description': scheme_description,
            'fixed_0_5_binary_metrics': fixed_metrics,
            'tuned_binary_metrics': tuned_metrics,
            'thresholds': thresholds,
            'test_average_precision_fixed_0_5': fixed_ap.to_dict(orient='records'),
            'test_average_precision_tuned': tuned_ap.to_dict(orient='records'),
            'test_prediction_counts_fixed_0_5': fixed_counts.to_dict(orient='records'),
            'test_prediction_counts_tuned': tuned_counts.to_dict(orient='records'),
            'fixed_classification_report': fixed_report,
            'tuned_classification_report': tuned_report,
        }

    # Save row-level test predictions for error analysis.
    pred_df = test_df[['key', 'SITUATION', 'REAPPRAISAL']].copy()
    for i, label in enumerate(LABEL_COLS):
        pred_df[f'{label}_true_score'] = test_labels[:, i]
        pred_df[f'{label}_score'] = test_probs[:, i]
        for scheme_name in BINARY_TARGET_SCHEMES:
            pred_df[f'{label}_{scheme_name}_true_binary'] = binary_target_matrix(test_df, scheme_name)[:, i]
            pred_df[f'{label}_{scheme_name}_pred_binary'] = tuned_predictions_by_scheme[scheme_name][:, i]

    prefix = output_dir / f'{config.experiment_name}_{run_id}'
    pd.DataFrame(history).to_csv(prefix.with_name(prefix.name + '_training_history.csv'), index=False)
    pred_df.to_csv(prefix.with_name(prefix.name + '_test_predictions.csv'), index=False)
    pd.concat(threshold_tables, ignore_index=True).to_csv(
        prefix.with_name(prefix.name + '_threshold_tuning.csv'), index=False
    )
    soft_metrics.to_csv(prefix.with_name(prefix.name + '_soft_metrics.csv'), index=False)
    pd.concat(count_tables, ignore_index=True).to_csv(
        prefix.with_name(prefix.name + '_prediction_counts.csv'), index=False
    )
    pd.concat(average_precision_tables, ignore_index=True).to_csv(
        prefix.with_name(prefix.name + '_average_precision.csv'), index=False
    )

    summary = {
        'experiment_name': config.experiment_name,
        'model_name': config.model_name,
        'run_id': run_id,
        'loss_mode': loss_mode,
        'binary_target_schemes': BINARY_TARGET_SCHEMES,
        'best_epoch': best_epoch,
        'best_val_loss': best_val_loss,
        'checkpoint_path': str(checkpoint_path),
        'config': asdict(config),
        'soft_summary': macro_soft_summary(soft_metrics),
        'binary_scheme_metrics': binary_scheme_metrics,
        # Majority is kept as top-level convenience because it is the main binary definition.
        'fixed_0_5_binary_metrics': binary_scheme_metrics['majority']['fixed_0_5_binary_metrics'],
        'tuned_binary_metrics': binary_scheme_metrics['majority']['tuned_binary_metrics'],
        'thresholds': binary_scheme_metrics['majority']['thresholds'],
        'test_soft_metrics': soft_metrics.to_dict(orient='records'),
    }
    write_json(prefix.with_name(prefix.name + '_metrics.json'), summary)
    return summary


## Learning-Rate Sweep Helpers


In [ ]:
def lr_slug(learning_rate: float) -> str:
    return f'{learning_rate:.0e}'.replace('-', '_').replace('+', '')


def scheduler_slug(scheduler_type: str) -> str:
    return scheduler_type.replace('_with_', '_').replace('_', '-')


def base_experiment_name(experiment_name: str) -> str:
    return experiment_name.split('_lr_')[0]


def with_tuning_settings(
    config: ModelConfig,
    learning_rate: float,
    num_epochs: int,
    patience: int,
    scheduler_type: str = 'linear',
) -> ModelConfig:
    # Linear keeps the original LR-tuning name. Other schedulers get distinct names to avoid overwrites.
    experiment_name = f'{config.experiment_name}_lr_{lr_slug(learning_rate)}'
    if scheduler_type != 'linear':
        experiment_name = f'{experiment_name}_sched_{scheduler_slug(scheduler_type)}'

    return ModelConfig(
        experiment_name=experiment_name,
        model_name=config.model_name,
        batch_size=config.batch_size,
        learning_rate=learning_rate,
        weight_decay=config.weight_decay,
        dropout=config.dropout,
        max_length=config.max_length,
        num_epochs=num_epochs,
        patience=patience,
        scheduler_type=scheduler_type,
        local_files_only=config.local_files_only,
        pooling=config.pooling,
    )


def build_lr_sweep_configs(
    base_configs: list[ModelConfig],
    learning_rates_by_model: dict[str, list[float]],
    num_epochs: int = EPOCHS_TO_TRY,
    patience: int = EARLY_STOPPING_PATIENCE,
    scheduler_type: str = 'linear',
) -> list[ModelConfig]:
    configs = []
    for base_config in base_configs:
        for learning_rate in learning_rates_by_model[base_config.experiment_name]:
            configs.append(with_tuning_settings(base_config, learning_rate, num_epochs, patience, scheduler_type))
    return configs


def build_scheduler_sweep_configs(
    base_config: ModelConfig,
    learning_rates: list[float],
    scheduler_types: list[str],
    num_epochs: int = EPOCHS_TO_TRY,
    patience: int = EARLY_STOPPING_PATIENCE,
) -> list[ModelConfig]:
    configs = []
    for learning_rate in learning_rates:
        for scheduler_type in scheduler_types:
            configs.append(with_tuning_settings(base_config, learning_rate, num_epochs, patience, scheduler_type))
    return configs


def result_prefix(config: ModelConfig, run_id: str, output_dir: Path = OUTPUT_DIR) -> Path:
    return output_dir / f'{config.experiment_name}_{run_id}'


def result_metrics_path(config: ModelConfig, run_id: str, output_dir: Path = OUTPUT_DIR) -> Path:
    prefix = result_prefix(config, run_id, output_dir)
    return prefix.with_name(prefix.name + '_metrics.json')


def load_existing_result(config: ModelConfig, run_id: str, output_dir: Path = OUTPUT_DIR) -> dict | None:
    path = result_metrics_path(config, run_id, output_dir)
    if not path.exists():
        return None
    return json.loads(path.read_text())


def summarize_runs(results: list[dict], output_dir: Path = OUTPUT_DIR, name: str = 'summary') -> pd.DataFrame:
    rows = []
    for result in results:
        config = result['config']
        row = {
            'experiment_name': result['experiment_name'],
            'base_experiment_name': base_experiment_name(result['experiment_name']),
            'model_name': result['model_name'],
            'run_id': result['run_id'],
            'loss_mode': result['loss_mode'],
            'learning_rate': config['learning_rate'],
            'scheduler_type': config.get('scheduler_type', 'linear'),
            'batch_size': config['batch_size'],
            'dropout': config['dropout'],
            'pooling': config.get('pooling', 'auto'),
            'num_epochs_planned': config['num_epochs'],
            'patience': config['patience'],
            'best_epoch': result['best_epoch'],
            'best_val_loss': result['best_val_loss'],
        }
        row.update(result['soft_summary'])
        for scheme_name, scheme_payload in result['binary_scheme_metrics'].items():
            row.update({
                f'{scheme_name}_fixed_{k}': v
                for k, v in scheme_payload['fixed_0_5_binary_metrics'].items()
            })
            row.update({
                f'{scheme_name}_tuned_{k}': v
                for k, v in scheme_payload['tuned_binary_metrics'].items()
            })
        rows.append(row)

    output_dir.mkdir(parents=True, exist_ok=True)
    summary_path = output_dir / f'{name}.csv'
    new_summary_df = pd.DataFrame(rows)
    if summary_path.exists():
        existing_summary_df = pd.read_csv(summary_path)
        if 'scheduler_type' not in existing_summary_df.columns:
            existing_summary_df['scheduler_type'] = 'linear'
        if 'pooling' not in existing_summary_df.columns:
            existing_summary_df['pooling'] = 'auto'
        summary_df = pd.concat([existing_summary_df, new_summary_df], ignore_index=True, sort=False)
        summary_df['scheduler_type'] = summary_df['scheduler_type'].fillna('linear')
        summary_df['pooling'] = summary_df['pooling'].fillna('auto')
        summary_df = summary_df.drop_duplicates(
            subset=['experiment_name', 'run_id', 'loss_mode', 'learning_rate', 'scheduler_type', 'pooling'],
            keep='last',
        )
    else:
        summary_df = new_summary_df
    summary_df.to_csv(summary_path, index=False)

    grouped = (
        summary_df.groupby(
            ['base_experiment_name', 'experiment_name', 'model_name', 'learning_rate', 'scheduler_type', 'pooling', 'loss_mode'],
            as_index=False,
        )
        .agg(
            best_val_loss_mean=('best_val_loss', 'mean'),
            best_val_loss_sd=('best_val_loss', 'std'),
            best_epoch_mean=('best_epoch', 'mean'),
            macro_mae_mean=('macro_mae', 'mean'),
            macro_mae_sd=('macro_mae', 'std'),
            mean_pearson_mean=('mean_pearson', 'mean'),
            mean_pearson_sd=('mean_pearson', 'std'),
            weak_presence_tuned_macro_f1_mean=('weak_presence_tuned_macro_f1', 'mean'),
            weak_presence_tuned_macro_f1_sd=('weak_presence_tuned_macro_f1', 'std'),
            majority_tuned_macro_f1_mean=('majority_tuned_macro_f1', 'mean'),
            majority_tuned_macro_f1_sd=('majority_tuned_macro_f1', 'std'),
            strong_presence_tuned_macro_f1_mean=('strong_presence_tuned_macro_f1', 'mean'),
            strong_presence_tuned_macro_f1_sd=('strong_presence_tuned_macro_f1', 'std'),
            n_runs=('run_id', 'count'),
        )
    )
    grouped.to_csv(output_dir / f'{name}_by_lr.csv', index=False)
    return summary_df


def run_random_split_suite(
    model_configs: list[ModelConfig],
    seeds: Iterable[int],
    loss_mode: str = 'binomial_nll',
    split_scheme: str = 'majority',
    output_dir: Path = OUTPUT_DIR,
    summary_name: str | None = None,
    skip_completed: bool = True,
) -> pd.DataFrame:
    results = []
    device = get_device()

    for seed in seeds:
        set_seed(seed)
        train_df, val_df, test_df = make_random_split(dataset, seed, split_scheme=split_scheme)

        for config in model_configs:
            set_seed(seed)
            run_id = f'random_seed{seed}'
            existing_result = load_existing_result(config, run_id, output_dir) if skip_completed else None
            if existing_result is not None:
                print(f"Skipping completed run: {config.experiment_name} | {run_id}")
                results.append(existing_result)
                continue

            results.append(
                run_single_experiment(
                    config,
                    train_df,
                    val_df,
                    test_df,
                    run_id=run_id,
                    output_dir=output_dir,
                    loss_mode=loss_mode,
                    device=device,
                )
            )

    if summary_name is None:
        summary_name = f'lr_epoch_tuning_{loss_mode}'
    return summarize_runs(results, output_dir, name=summary_name)


def run_leave_one_vignette_suite(
    model_configs: list[ModelConfig],
    seed: int = 42,
    loss_mode: str = 'binomial_nll',
    split_scheme: str = 'majority',
    output_dir: Path = OUTPUT_DIR,
) -> pd.DataFrame:
    results = []
    device = get_device()

    for vignette_idx, (situation, train_df, val_df, test_df) in enumerate(
        make_leave_one_vignette_splits(dataset, seed, split_scheme=split_scheme)
    ):
        for config in model_configs:
            set_seed(seed)
            run_id = f'leave_vignette_{vignette_idx}_seed{seed}'
            results.append(
                run_single_experiment(
                    config,
                    train_df,
                    val_df,
                    test_df,
                    run_id=run_id,
                    output_dir=output_dir,
                    loss_mode=loss_mode,
                    device=device,
                )
            )

    return summarize_runs(results, output_dir, name=f'leave_one_vignette_lr_epoch_tuning_{loss_mode}')


## Quick Smoke Test


In [ ]:
SMOKE_TEST = False

if SMOKE_TEST:
    smoke_config = with_tuning_settings(
        BERT_BASE_CONFIG,
        learning_rate=1e-5,
        num_epochs=1,
        patience=1,
    )
    train_df, val_df, test_df = make_random_split(dataset, seed=42)
    smoke_result = run_single_experiment(
        smoke_config,
        train_df,
        val_df,
        test_df,
        run_id='smoke_seed42',
        loss_mode='binomial_nll',
    )
    display(smoke_result['soft_summary'])


## Run One Model At A Time

Use this cell as a launcher. Pick a single model and either one learning rate or a small list of learning rates. This avoids accidentally running the full BERT-base, BERT-large, and ModernBERT sweep in one notebook execution.


In [ ]:
SEEDS = [13]

MODEL_CONFIG_BY_NAME = {
    'bert_base_uncased': BERT_BASE_CONFIG,
    'bert_large_uncased': BERT_LARGE_CONFIG,
    'modernbert_base': MODERNBERT_CONFIG,
    'roberta_base': ROBERTA_BASE_CONFIG,
    'roberta_large': ROBERTA_LARGE_CONFIG,
}

# RoBERTa comparison against the current best BERT-large setup.
MODEL_TO_RUN = 'roberta_large'
LEARNING_RATES_TO_RUN = [1e-5]
SCHEDULERS_TO_RUN = ['linear']

# Safe defaults: training is opt-in and completed configurations are not overwritten.
RUN_SELECTED_TUNING = False
SKIP_COMPLETED_TUNING = True

SELECTED_MODEL_CONFIGS = build_scheduler_sweep_configs(
    MODEL_CONFIG_BY_NAME[MODEL_TO_RUN],
    learning_rates=LEARNING_RATES_TO_RUN,
    scheduler_types=SCHEDULERS_TO_RUN,
    num_epochs=EPOCHS_TO_TRY,
    patience=EARLY_STOPPING_PATIENCE,
)

print(f'Planned training configs for this launch: {len(SELECTED_MODEL_CONFIGS)}')
display(pd.DataFrame([asdict(config) for config in SELECTED_MODEL_CONFIGS]))

if RUN_SELECTED_TUNING:
    tuning_summary = run_random_split_suite(
        model_configs=SELECTED_MODEL_CONFIGS,
        seeds=SEEDS,
        loss_mode='binomial_nll',
        summary_name='lr_epoch_tuning_binomial_nll',
        skip_completed=SKIP_COMPLETED_TUNING,
    )
    display(
        tuning_summary.sort_values(
            ['best_val_loss', 'macro_mae', 'mean_pearson'],
            ascending=[True, True, False],
        )
    )
else:
    print('Set RUN_SELECTED_TUNING = True when you are ready to launch only the selected scheduler block.')


## Optional Leave-One-Vignette-Out Robustness Check

After selecting a promising encoder and learning rate from the random split, use this section for a stronger robustness check across vignette-held-out test sets.


In [ ]:
RUN_LEAVE_ONE_VIGNETTE = False

if RUN_LEAVE_ONE_VIGNETTE:
    # Replace this with the selected/best config after inspecting the tuning summary.
    best_config = SELECTED_MODEL_CONFIGS[0]
    vignette_summary = run_leave_one_vignette_suite(
        model_configs=[best_config],
        seed=42,
        loss_mode='binomial_nll',
    )
    display(vignette_summary)


## Validation-Loss Curves And LR Comparison


In [ ]:
# Load saved tuning result files. This works after training, and also when reopening the notebook later.
import os

MPL_CACHE_DIR = PROJECT_DIR / '.matplotlib_cache'
MPL_CACHE_DIR.mkdir(exist_ok=True)
os.environ.setdefault('MPLCONFIGDIR', str(MPL_CACHE_DIR))

import matplotlib.pyplot as plt

plt.rcParams.update({
    'figure.figsize': (9, 5),
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.25,
    'font.size': 11,
})

SUMMARY_PATH = OUTPUT_DIR / 'lr_epoch_tuning_binomial_nll.csv'
SUMMARY_BY_LR_PATH = OUTPUT_DIR / 'lr_epoch_tuning_binomial_nll_by_lr.csv'

if SUMMARY_PATH.exists():
    tuning_summary = pd.read_csv(SUMMARY_PATH)
    if 'scheduler_type' not in tuning_summary.columns:
        tuning_summary['scheduler_type'] = 'linear'
    tuning_summary['scheduler_type'] = tuning_summary['scheduler_type'].fillna('linear')
    tuning_summary['lr_label'] = tuning_summary['learning_rate'].map(lambda lr: f'{lr:.0e}')
    display(
        tuning_summary[[
            'base_experiment_name',
            'lr_label',
            'scheduler_type',
            'best_epoch',
            'best_val_loss',
            'macro_mae',
            'macro_rmse',
            'mean_pearson',
            'majority_tuned_macro_f1',
            'strong_presence_tuned_macro_f1',
        ]]
        .sort_values(['best_val_loss', 'macro_mae', 'mean_pearson'], ascending=[True, True, False])
        .round(4)
    )
else:
    print('No tuning summary yet:', SUMMARY_PATH)


In [ ]:
def load_training_histories(output_dir: Path = OUTPUT_DIR) -> pd.DataFrame:
    history_files = sorted(output_dir.glob('*_training_history.csv'))
    histories = []
    for path in history_files:
        df = pd.read_csv(path)
        if {'experiment_name', 'learning_rate', 'epoch', 'train_loss', 'val_loss'}.issubset(df.columns):
            df['source_file'] = path.name
            df['base_experiment_name'] = df['experiment_name'].str.split('_lr_').str[0]
            if 'scheduler_type' not in df.columns:
                df['scheduler_type'] = 'linear'
            df['scheduler_type'] = df['scheduler_type'].fillna('linear')
            df['lr_label'] = df['learning_rate'].map(lambda lr: f'{lr:.0e}')
            df['condition_label'] = df['lr_label'] + ' | ' + df['scheduler_type']
            histories.append(df)
    if not histories:
        return pd.DataFrame()
    return pd.concat(histories, ignore_index=True)


def plot_validation_loss_curves(history_df: pd.DataFrame, base_experiment_name: str | None = None) -> None:
    if history_df.empty:
        print('No training histories found yet.')
        return

    plot_df = history_df.copy()
    if base_experiment_name is not None:
        plot_df = plot_df[plot_df['base_experiment_name'] == base_experiment_name]

    for model_name, model_df in plot_df.groupby('base_experiment_name'):
        fig, ax = plt.subplots(figsize=(9, 5))
        for (experiment_name, run_id), run_df in model_df.groupby(['experiment_name', 'run_id']):
            run_df = run_df.sort_values('epoch')
            condition_label = run_df['condition_label'].iloc[0]
            best_idx = run_df['val_loss'].idxmin()
            ax.plot(run_df['epoch'], run_df['val_loss'], marker='o', linewidth=2, label=condition_label)
            ax.scatter(
                run_df.loc[best_idx, 'epoch'],
                run_df.loc[best_idx, 'val_loss'],
                s=70,
                zorder=3,
            )
        ax.set_title(f'Validation Loss During Fine-Tuning: {model_name}')
        ax.set_xlabel('Epoch')
        ax.set_ylabel('Validation loss')
        ax.legend(title='LR | scheduler')
        plt.tight_layout()
        plt.show()


def plot_train_vs_val_for_best_run(history_df: pd.DataFrame, summary_df: pd.DataFrame) -> None:
    if history_df.empty or summary_df.empty:
        print('Need both training histories and summary results first.')
        return

    best = summary_df.sort_values(['best_val_loss', 'macro_mae', 'mean_pearson'], ascending=[True, True, False]).iloc[0]
    run_df = history_df[
        (history_df['experiment_name'] == best['experiment_name'])
        & (history_df['run_id'] == best['run_id'])
    ].sort_values('epoch')

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(run_df['epoch'], run_df['train_loss'], marker='o', label='Training loss')
    ax.plot(run_df['epoch'], run_df['val_loss'], marker='o', label='Validation loss')
    ax.axvline(best['best_epoch'], color='black', linestyle='--', alpha=0.55, label=f"best epoch {int(best['best_epoch'])}")
    scheduler_type = best.get('scheduler_type', 'linear')
    ax.set_title(f"Best Run Learning Curve: {best['base_experiment_name']} lr={best['learning_rate']:.0e} {scheduler_type}")
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.legend()
    plt.tight_layout()
    plt.show()


history_df = load_training_histories(OUTPUT_DIR)
plot_validation_loss_curves(history_df)

if SUMMARY_PATH.exists():
    plot_train_vs_val_for_best_run(history_df, tuning_summary)


In [ ]:
if SUMMARY_BY_LR_PATH.exists():
    summary_by_lr = pd.read_csv(SUMMARY_BY_LR_PATH)
    summary_by_lr['lr_label'] = summary_by_lr['learning_rate'].map(lambda lr: f'{lr:.0e}')
    if 'scheduler_type' not in summary_by_lr.columns:
        summary_by_lr['scheduler_type'] = 'linear'
    summary_by_lr['scheduler_type'] = summary_by_lr['scheduler_type'].fillna('linear')
    summary_by_lr['condition_label'] = summary_by_lr['lr_label'] + ' | ' + summary_by_lr['scheduler_type']

    display(
        summary_by_lr[[
            'base_experiment_name',
            'lr_label',
            'scheduler_type',
            'best_epoch_mean',
            'best_val_loss_mean',
            'macro_mae_mean',
            'mean_pearson_mean',
            'majority_tuned_macro_f1_mean',
            'n_runs',
        ]]
        .sort_values(['best_val_loss_mean', 'macro_mae_mean', 'mean_pearson_mean'], ascending=[True, True, False])
        .round(4)
    )

    for metric, ylabel, title in [
        ('best_val_loss_mean', 'Best validation loss', 'Best Validation Loss by Learning Rate'),
        ('macro_mae_mean', 'Macro MAE', 'Test Macro MAE by Learning Rate'),
        ('mean_pearson_mean', 'Mean Pearson', 'Soft-Label Correlation by Learning Rate'),
    ]:
        fig, ax = plt.subplots(figsize=(9, 4.5))
        for model_name, model_df in summary_by_lr.groupby('base_experiment_name'):
            model_df = model_df.sort_values(['learning_rate', 'scheduler_type'])
            ax.plot(model_df['condition_label'], model_df[metric], marker='o', linewidth=2, label=model_name)
        ax.set_title(title)
        ax.set_xlabel('Learning rate | scheduler')
        ax.set_ylabel(ylabel)
        ax.legend(title='Model')
        plt.tight_layout()
        plt.show()
else:
    print('No grouped LR summary yet:', SUMMARY_BY_LR_PATH)


## Select Candidate For Final Training


In [ ]:
if SUMMARY_PATH.exists():
    if 'scheduler_type' not in tuning_summary.columns:
        tuning_summary['scheduler_type'] = 'linear'
    tuning_summary['scheduler_type'] = tuning_summary['scheduler_type'].fillna('linear')

    # Primary selection criterion is validation loss because this is the held-out signal used during training.
    # Macro MAE and correlation are shown so you can check that the selected LR also improves test soft-label behavior.
    selection_table = (
        tuning_summary[[
            'base_experiment_name',
            'experiment_name',
            'model_name',
            'learning_rate',
            'scheduler_type',
            'batch_size',
            'dropout',
            'num_epochs_planned',
            'patience',
            'best_epoch',
            'best_val_loss',
            'macro_mae',
            'mean_pearson',
            'majority_tuned_macro_f1',
            'strong_presence_tuned_macro_f1',
        ]]
        .sort_values(['best_val_loss', 'macro_mae', 'mean_pearson'], ascending=[True, True, False])
        .reset_index(drop=True)
    )
    display(selection_table.round(4))

    best_row = selection_table.iloc[0]
    print('Recommended next candidate:')
    print(
        f"{best_row['base_experiment_name']} at lr={best_row['learning_rate']:.0e}, scheduler={best_row.get('scheduler_type', 'linear')}; "
        f"best epoch={int(best_row['best_epoch'])}; "
        f"best validation loss={best_row['best_val_loss']:.4f}; "
        f"macro MAE={best_row['macro_mae']:.4f}."
    )
else:
    print('Run the tuning sweep first, then rerun this cell.')
